In [10]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# ============================================================
# 1. Input data
# ============================================================

data = [
    ("charlie kirk",3,1800),
    ("2026 winter olympics",1,1300),
    ("iran",4,350),
    ("67",2,300),
    ("silver price today",1,250),
    ("gemini",3,190),
    ("gold price",2,170),
    ("chatgpt",12,130),
    ("news today",2,70),
    ("google games",1,70),
    ("knicks",2,60),
    ("ai detector",1,50),
    ("chat gpt",5,50),
    ("tiempo de mañana",1,40),
    ("weather tomorrow",6,40),
    ("dow jones",2,30),
    ("warriors",2,30),
    ("dodgers",3,30),
    ("linkedin",4,30),
    ("real madrid",2,30),
    ("detroit tigers",1,30),
    ("coffee near me",1,20),
    ("email",8,20),
    ("roblox",5,20),
    ("canva",4,20),
    ("clima",2,20),
    ("gm",1,20),
    ("instagram",9,20),
    ("dow jones today",1,20),
    ("yt",1,20),
    ("maps",6,20),
    ("expedia",1,20),
    ("blooket join",1,20),
    ("doordash",2,20),
    ("nfl schedule",1,10),
    ("nfl",14,10),
    ("people",18,10),
    ("docs",4,10),
    ("bitcoin price",1,10),
    ("champions league",1,10),
    ("blooket",4,10),
    ("fedex",3,10),
    ("yahoo finance",1,10),
    ("hobby lobby",2,10),
    ("premier league",1,10),
    ("discord",2,10),
    ("spotify",4,10),
    ("outlook",4,10),
    ("you",100,10),
    ("mlb",6,10),
]


df = pd.DataFrame(
    data,
    columns=["query", "search_interest", "increase_percent"]
)


In [11]:

# ============================================================
# 2. Text feature: TF-IDF embedding
# ============================================================

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2)
)

text_features = tfidf.fit_transform(df["query"])

text_features = text_features.toarray()



In [12]:
# ============================================================
# 3. Numerical features
# ============================================================

# log transform because increase_percent has huge outliers
df["log_interest"] = np.log1p(df["search_interest"])

df["log_growth"] = np.sign(df["increase_percent"]) * np.log1p(
    np.abs(df["increase_percent"])
)


numeric_features = df[
    [
        "log_interest",
        "log_growth"
    ]
].values


# Standardize numerical part
scaler = StandardScaler()

numeric_features = scaler.fit_transform(
    numeric_features
)



In [13]:

# ============================================================
# 4. Hybrid feature vector
# ============================================================

# combine:
# [semantic query vector | popularity | growth]

hybrid_features = np.hstack(
    [
        text_features,
        numeric_features
    ]
)


print(
    "Hybrid vector dimension:",
    hybrid_features.shape
)

Hybrid vector dimension: (50, 91)


In [14]:
# ============================================================
# 5. Cluster
# ============================================================

# Try several k values
scores = {}

for k in range(2,10):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(
        hybrid_features
    )

    scores[k] = silhouette_score(
        hybrid_features,
        labels
    )


best_k = max(
    scores,
    key=scores.get
)

print(
    "Best k:",
    best_k,
    "Silhouette:",
    scores[best_k]
)


# final clustering

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)


df["cluster"] = kmeans.fit_predict(
    hybrid_features
)

Best k: 2 Silhouette: 0.3222843039097221


In [15]:
# ============================================================
# 6. Display clusters
# ============================================================

for c in sorted(df.cluster.unique()):

    print("\n====================")
    print("Cluster", c)
    print("====================")

    print(
        df[
            df.cluster == c
        ][
            [
                "query",
                "search_interest",
                "increase_percent"
            ]
        ]
        .sort_values(
            "increase_percent",
            ascending=False
        )
        .to_string(index=False)
    )


Cluster 0
           query  search_interest  increase_percent
      news today                2                70
    google games                1                70
          knicks                2                60
     ai detector                1                50
        chat gpt                5                50
tiempo de mañana                1                40
weather tomorrow                6                40
       dow jones                2                30
        warriors                2                30
         dodgers                3                30
        linkedin                4                30
     real madrid                2                30
  detroit tigers                1                30
  coffee near me                1                20
           email                8                20
          roblox                5                20
           canva                4                20
           clima                2                20
 

In [16]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize


# ============================================================
# 1. Data
# ============================================================

data = [
    ("charlie kirk",3,1800),
    ("2026 winter olympics",1,1300),
    ("iran",4,350),
    ("67",2,300),
    ("silver price today",1,250),
    ("gemini",3,190),
    ("gold price",2,170),
    ("chatgpt",12,130),
    ("news today",2,70),
    ("google games",1,70),
    ("knicks",2,60),
    ("ai detector",1,50),
    ("chat gpt",5,50),
    ("weather tomorrow",6,40),
    ("dow jones",2,30),
    ("warriors",2,30),
    ("dodgers",3,30),
    ("linkedin",4,30),
    ("real madrid",2,30),
    ("detroit tigers",1,30),
    ("coffee near me",1,20),
    ("email",8,20),
    ("roblox",5,20),
    ("canva",4,20),
    ("clima",2,20),
    ("gm",1,20),
    ("instagram",9,20),
    ("dow jones today",1,20),
    ("yt",1,20),
    ("maps",6,20),
    ("expedia",1,20),
    ("blooket join",1,20),
    ("doordash",2,20),
    ("nfl schedule",1,10),
    ("nfl",14,10),
    ("people",18,10),
    ("docs",4,10),
    ("bitcoin price",1,10),
    ("champions league",1,10),
    ("blooket",4,10),
    ("fedex",3,10),
    ("yahoo finance",1,10),
    ("hobby lobby",2,10),
    ("premier league",1,10),
    ("discord",2,10),
    ("spotify",4,10),
    ("outlook",4,10),
    ("you",100,10),
    ("mlb",6,10),
]


df = pd.DataFrame(
    data,
    columns=[
        "query",
        "search_interest",
        "increase_percent"
    ]
)


# ============================================================
# 2. Sentence-BERT semantic embeddings
# ============================================================

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


query_embeddings = model.encode(
    df["query"].tolist(),
    show_progress_bar=True
)


# normalize embeddings
query_embeddings = normalize(
    query_embeddings
)


print(
    "Embedding dimension:",
    query_embeddings.shape
)


# ============================================================
# 3. Attention features
# ============================================================

# log transform to reduce extreme shocks
df["log_interest"] = np.log1p(
    df["search_interest"]
)

df["log_growth"] = (
    np.sign(df["increase_percent"])
    *
    np.log1p(
        np.abs(df["increase_percent"])
    )
)


attention_features = df[
    [
        "log_interest",
        "log_growth"
    ]
].values


attention_features = StandardScaler().fit_transform(
    attention_features
)


# ============================================================
# 4. Hybrid vector
# ============================================================

# increase this if you want clusters to emphasize shocks
shock_weight = 3


hybrid_features = np.hstack(
    [
        query_embeddings,
        shock_weight * attention_features
    ]
)


print(
    "Hybrid vector:",
    hybrid_features.shape
)


# ============================================================
# 5. Find optimal number of clusters
# ============================================================

silhouette_scores = {}


for k in range(2,10):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = kmeans.fit_predict(
        hybrid_features
    )

    score = silhouette_score(
        hybrid_features,
        labels
    )

    silhouette_scores[k] = score


best_k = max(
    silhouette_scores,
    key=silhouette_scores.get
)


print(
    "Best cluster number:",
    best_k
)

print(
    "Silhouette:",
    silhouette_scores[best_k]
)


# ============================================================
# 6. Final clustering
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=50
)


df["cluster"] = kmeans.fit_predict(
    hybrid_features
)


# ============================================================
# 7. Inspect clusters
# ============================================================

for c in sorted(df.cluster.unique()):

    print("\n")
    print("="*50)
    print("CLUSTER", c)
    print("="*50)

    print(
        df[
            df.cluster == c
        ][
            [
                "query",
                "search_interest",
                "increase_percent"
            ]
        ]
        .sort_values(
            "increase_percent",
            ascending=False
        )
        .to_string(index=False)
    )

Batches: 100%|██████████| 2/2 [00:00<00:00, 10.97it/s]


Embedding dimension: (49, 384)
Hybrid vector: (49, 386)
Best cluster number: 2
Silhouette: 0.45944936213844534


CLUSTER 0
           query  search_interest  increase_percent
      news today                2                70
    google games                1                70
          knicks                2                60
     ai detector                1                50
        chat gpt                5                50
weather tomorrow                6                40
       dow jones                2                30
        warriors                2                30
         dodgers                3                30
        linkedin                4                30
     real madrid                2                30
  detroit tigers                1                30
  coffee near me                1                20
           email                8                20
          roblox                5                20
           canva                4            

In [21]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


selected_queries = []


for cluster_id in sorted(df.cluster.unique()):

    # indices belonging to cluster
    idx = df[
        df.cluster == cluster_id
    ].index


    cluster_embeddings = embeddings[idx]


    # centroid
    centroid = (
        cluster_embeddings
        .mean(axis=0)
        .reshape(1,-1)
    )


    # similarity to centroid
    similarity = cosine_similarity(
        cluster_embeddings,
        centroid
    ).flatten()


    # rank closest to centroid
    ranked = (
        pd.DataFrame({
            "query":
                df.loc[idx,"query"].values,

            "similarity":
                similarity,

            "search_interest":
                df.loc[idx,"search_interest"].values,

            "increase_percent":
                df.loc[idx,"increase_percent"].values
        })
        .sort_values(
            "similarity",
            ascending=False
        )
    )


    print("\n")
    print("="*40)
    print("CLUSTER", cluster_id)
    print("="*40)

    print(
        ranked.head(5)
    )


    selected_queries.extend(
        ranked.head(5)["query"]
        .tolist()
    )



CLUSTER 0
               query  similarity  search_interest  increase_percent
40               mlb    0.631367                6                10
26               nfl    0.612913               14                10
35    premier league    0.611417                1                10
17                gm    0.581875                1                20
30  champions league    0.562140                1                10


CLUSTER 1
                query  similarity  search_interest  increase_percent
3                  67    0.600807                2               300
6          gold price    0.598892                2               170
5              gemini    0.558003                3               190
4  silver price today    0.547566                1               250
2                iran    0.514969                4               350
